# 05 · Chatbot RAG con memoria conversacional

**Taller Práctico Partes 3 y 4 — Integración RAG + LLM + Memoria**

Integramos el recuperador FAISS con el LLM `QWEN\QWEN3.5-27B` y la memoria
conversacional:
1. `RAGService` → recupera contexto de los catálogos.
2. `ChatbotService` → agente autónomo que responde en JSON estructurado
   y ejecuta acciones (registrar venta, consultar inventario/ventas).
3. `MemoryManager` → mantiene el contexto del diálogo multi-turno.

> Requiere haber indexado los catálogos (notebook 03).

In [1]:
import sys
from pathlib import Path

# Asegura que el notebook encuentre los módulos del proyecto (config, rag_service, ...)
sys.path.insert(0, str(Path.cwd().parent))


In [2]:
# import database
# database.ensure_data_files()

In [3]:
from config import Config
from rag_service import RAGService
from llm_service import ChatbotService

# 1. RAG: cargar índice de catálogos
rag = RAGService()
if not rag.load_index():
    print("Ejecuta primero el notebook 03 (indexación).")

# 2. Chatbot: LLM qwen-plus + memoria por usuario
chatbot = ChatbotService(rag)
print("✓ Chatbot listo")

Inicializando servicio RAG...
  Proveedor de embeddings: OPEN AI API
  Embeddings OPEN AI: text-embedding-3-small (dim=1536, batch=10)
Servicio RAG inicializado

Cargando índice desde: c:\Users\rmend\Dropbox\Personal_Desktop2026\BootCamp ML\DMC\Diplomado_AI_Engineering\DesignImplementation_Chatbots\Chatbot-AIGenerative-BankingSystem\data\faiss_index
Índice cargado exitosamente
Inicializando servicio de chatbot...
  Modelo: Qwen/Qwen3.5-9B
  API Base: https://router.huggingface.co/v1
Servicio de chatbot inicializado

✓ Chatbot listo


In [4]:
# 3. CONVERSACIÓN MULTI-TURNO (memoria en acción)
user_id = "demo_notebook"
chatbot.clear_memory(user_id)

preguntas = [
    "cual es el fraude financiero más común en el sistema bancario peruano?"
]

for msg in preguntas:
    resp = chatbot.process_message(user_id, msg)
    print(f"👤 {msg}")
    print(f"🤖 {resp.get('to_user', resp)}\n")

👤 cual es el fraude financiero más común en el sistema bancario peruano?
🤖 Como asistente virtual de INDECOPI, debo informarle que, basándome en la información disponible en las resoluciones que tengo a mi disposición, no existe un reporte estadístico general que clasifique explícitamente cuál es el "fraude financiero más común" en términos de frecuencia absoluta o porcentaje total en todo el sistema bancario peruano. Sin embargo, sí puedo identificar con claridad el tipo de conducta que genera la mayor cantidad de denuncias y sanciones ante nuestra institución en el último periodo analizado.

El patrón de infracción que aparece con mayor recurrencia en los expedientes resueltos es el de **operaciones no reconocidas realizadas mediante tarjetas de crédito o débito**, las cuales se ejecutan sin el consentimiento del titular de la cuenta. Este tipo de hechos suelen ocurrir cuando las entidades financieras no adoptan las medidas de seguridad pertinentes para impedir transacciones que no c